# 04 ADS-B movement

Turns the filtered ADS-B positions from `03_adsb_fetch` into one trajectory per departure and finds the moment the
aircraft first starts to move on the ground. Also measures how far that point is from the stand and from the runway
threshold.

On 32 cores this takes several hours; it was run as a Slurm batch job with
`jupyter nbconvert --to notebook --execute`. Results are never overwritten, so each run has its own `RUN_NAME`.

In [1]:
RUN_NAME = "movement_1s_v5"
MAX_WORKERS = 32
RAM_BUDGET_GIB = 96        # planning budget for the worker count, not a hard limit
N_BUCKETS = 512

In [2]:
import os
for variable in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[variable] = "1"
import json
import sys
import uuid
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT / "data"))
import adsb_cleanup_worker as workflow      # spawn workers need importable modules
import adsb_extra_features as extra

config = workflow.Config(max_workers=MAX_WORKERS, ram_budget_gib=RAM_BUDGET_GIB, n_buckets=N_BUCKETS)
SOURCES = sorted((ROOT / "data/04_additional_data/adsb_lol").glob("????-??.parquet"))
PROCESSED = ROOT / "data/04_additional_data/adsb_lol_processed" / RUN_NAME
FEATURES = ROOT / "data/03_features" / f"adsb_movement_{RUN_NAME}.parquet"
SCRATCH = Path(os.environ.get("SLURM_TMPDIR", ROOT / "data/04_additional_data/.adsb_prepare_tmp"))
assert not PROCESSED.exists() and not FEATURES.exists(), f"{RUN_NAME} exists already; choose a new RUN_NAME"
display(workflow.inspect_sources(SOURCES))
workflow.resource_report(config)

,file,rows,bytes,parquet_uncompressed_bytes
0,2024-12.parquet,0,1713,0
1,2025-01.parquet,12378515,225618101,303414779
2,2025-02.parquet,11506193,208742609,280363028
3,2025-03.parquet,14715531,265159596,356434888
4,2025-04.parquet,14598531,262299205,353212338
5,2025-05.parquet,18804458,335084785,453064766
6,2025-06.parquet,22113183,385877998,524356280
7,2025-07.parquet,20734636,365314796,497666750
8,2025-08.parquet,21577197,377863422,514470187
9,2025-09.parquet,21039894,369191523,503017530


{'cpu_affinity': [9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120],
 'cpu_count': 192,
 'SLURM_CPUS_PER_TASK': '32',
 'configured_MAX_WORKERS': 32,
 'effective_workers': 29,
 'available_ram_gib': 2108.6728591918945,
 'process_rss_gib': 0.21870040893554688,
 'N_BUCKETS': 512}

## Stand and runway positions

Matches every departure's stand and runway to coordinates (OpenStreetMap stands, X-Plane runways), so the distances
of the first movement point can be measured later. Runs first because it is quick and fails early on bad input.

In [3]:
context = FEATURES.parent / f".{FEATURES.stem}.context-{uuid.uuid4().hex}"
# ranking.parquet is left out: its January/July rows are the final ranking rows with slightly shifted times.
CLEAN_FILES = [ROOT / "data/02_clean" / name for name in
               ["train_DEP_merged_sorted.parquet", "train_ARR_merged_sorted.parquet", "final_ranking.parquet"]]
extra.build_context(CLEAN_FILES, context, config.n_buckets)
print("Matching audit:", context.relative_to(ROOT))

Clean metadata: train_DEP_merged_sorted.parquet


Clean metadata: train_ARR_merged_sorted.parquet


Clean metadata: final_ranking.parquet


Geometry: EDDF, matching 256 stands


Geometry EDDF: {'exact': 167, 'normalized_ref': 41, 'mean_same_ref': 20, 'missing': 16, 'parent_stand_estimate': 6, 'xplane_fallback': 3, 'mean_variants': 1, 'ambiguous_spread': 1, 'gate_estimate': 1}


...

Geometry metadata: 416/512 buckets saved


Geometry metadata: 448/512 buckets saved


Geometry metadata: 480/512 buckets saved


Geometry metadata: 512/512 buckets saved


Matching audit: data/03_features/.adsb_movement_movement_1s_v5.context-ab48871f516c4900bdb1e1b200cf9646


## Stage A: one trajectory per departure

Positions are grouped by departure, split into physical flights with `traffic`, and the flight that contains the
take-off (or starts right after it) is kept and resampled to one second. Ties stay unresolved instead of guessed.

In [4]:
stage_a = workflow.stage_a(SOURCES, PROCESSED, SCRATCH, config)
pd.Series(stage_a["totals"])

{
  "cpu_affinity": [
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    105,
    106,
    107,
    108,
    109,
    110,
    111,
    112,
    113,
    114,
    115,
    116,
    117,
    118,
    119,
    120
  ],
  "cpu_count": 192,
  "SLURM_CPUS_PER_TASK": "32",
  "configured_MAX_WORKERS": 32,
  "effective_workers": 29,
  "available_ram_gib": 2107.8302688598633,
  "process_rss_gib": 0.3297004699707031,
  "N_BUCKETS": 512
}
Hash pass: 2024-12.parquet; 0 rows staged


Hash pass: 2025-01.parquet; 12,378,515 rows staged


Hash pass: 2025-02.parquet; 23,884,708 rows staged


Hash pass: 2025-03.parquet; 38,600,239 rows staged


Hash pass: 2025-04.parquet; 53,198,770 rows staged


...

{'buckets': '510/512', 'mvts': 2336952, 'assigned_mvts': 2331920, 'single': 2211162, 'multiple': 120758, 'physical_flights': 2460027, 'max_physical_flights': 8, 'selected': 2308409, 'resampled': 2308409, 'rows': 1293641653, 'unresolved': 28543, 'contains_mvt': 1494622, 'starts_after_mvt': 811514, 'ends_before_mvt': 2273, 'elapsed_sec': 3468.1, 'mvts_per_sec': 673.84, 'rss_gib': 2.1533203125}


{'buckets': '511/512', 'mvts': 2341486, 'assigned_mvts': 2336444, 'single': 2215452, 'multiple': 120992, 'physical_flights': 2464801, 'max_physical_flights': 8, 'selected': 2312876, 'resampled': 2312876, 'rows': 1296163238, 'unresolved': 28610, 'contains_mvt': 1497512, 'starts_after_mvt': 813080, 'ends_before_mvt': 2284, 'elapsed_sec': 3470.6, 'mvts_per_sec': 674.67, 'rss_gib': 2.1533203125}


{'buckets': '512/512', 'mvts': 2346159, 'assigned_mvts': 2341105, 'single': 2219857, 'multiple': 121248, 'physical_flights': 2469729, 'max_physical_flights': 8, 'selected': 2317493, 'resampled': 2317493, 'rows': 1298697638, 'unresolved': 28666, 'contains_mvt': 1500419, 'starts_after_mvt': 814787, 'ends_before_mvt': 2287, 'elapsed_sec': 3475.9, 'mvts_per_sec': 674.97, 'rss_gib': 2.1533203125}


Stage A committed: movement_1s_v5; Parquet size 44.467 GiB


mvts                       2346159
assigned_mvts              2341105
single                     2219857
multiple                    121248
physical_flights           2469729
max_physical_flights             8
selected                   2317493
resampled                  2317493
rows                    1298697638
unresolved                   28666
contains_mvt               1500419
starts_after_mvt            814787
ends_before_mvt               2287
dtype: int64

## Stage B: first movement and distances

`Flight.movement()` (speed above 2 kn for at least 30 s) gives the start of the first sustained movement. The table
keeps absolute times, so `05_features` can express them relative to any dataset's take-off time.

In [5]:
stage_b = workflow.stage_b(PROCESSED, FEATURES, config, partition_function=extra.feature_partition,
                           partition_args=(str(context),), feature_schema=extra.SCHEMA, extra_validator=extra.validate_extra)
workflow.json_write(FEATURES.with_suffix(".json"), {**stage_b, "geometry_context": str(context)})
pd.Series(stage_b)

{'stage': 'B', 'buckets': '0/512', 'mvts': 0, 'detections': 0, 'elapsed_sec': 30.1, 'mvts_per_sec': 0.0, 'rss_gib': 2.160411834716797}


{'stage': 'B', 'buckets': '0/512', 'mvts': 0, 'detections': 0, 'elapsed_sec': 60.1, 'mvts_per_sec': 0.0, 'rss_gib': 2.1604156494140625}


{'stage': 'B', 'buckets': '0/512', 'mvts': 0, 'detections': 0, 'elapsed_sec': 90.1, 'mvts_per_sec': 0.0, 'rss_gib': 2.1604156494140625}


{'stage': 'B', 'buckets': '0/512', 'mvts': 0, 'detections': 0, 'elapsed_sec': 120.1, 'mvts_per_sec': 0.0, 'rss_gib': 2.1604156494140625}


{'stage': 'B', 'buckets': '0/512', 'mvts': 0, 'detections': 0, 'elapsed_sec': 150.1, 'mvts_per_sec': 0.0, 'rss_gib': 2.1604156494140625}


...

{'stage': 'B', 'buckets': '509/512', 'mvts': 2332450, 'detections': 2274630, 'elapsed_sec': 3776.1, 'mvts_per_sec': 617.68, 'rss_gib': 2.1868858337402344}


{'stage': 'B', 'buckets': '510/512', 'mvts': 2337029, 'detections': 2279100, 'elapsed_sec': 3776.9, 'mvts_per_sec': 618.77, 'rss_gib': 2.1868858337402344}


{'stage': 'B', 'buckets': '511/512', 'mvts': 2341625, 'detections': 2283603, 'elapsed_sec': 3777.2, 'mvts_per_sec': 619.94, 'rss_gib': 2.1868858337402344}


{'stage': 'B', 'buckets': '512/512', 'mvts': 2346159, 'detections': 2288020, 'elapsed_sec': 3787.1, 'mvts_per_sec': 619.51, 'rss_gib': 2.1868858337402344}


stage                                                B
processed                               movement_1s_v5
rows                                           2346159
valid_movement_detections                      2288020
failed_movement_detections                       29473
unresolved                                       28666
coverage_percent                              97.52195
speed_threshold                                      2
time_threshold                                     30s
resample_rule                                       1s
traffic_version               2.13.post75.dev0+aee9ba0
bytes                                        353421334
dtype: object

## Coverage

In [6]:
report = workflow.diagnostics(PROCESSED, FEATURES)
display(pd.Series(report["totals"]))
display(report["coverage"]["airport"])
display(report["coverage"]["month"])
report["distribution"]

total_raw_mvts                2.346159e+06
assigned_mvts                 2.341105e+06
multiple_flights              1.212480e+05
unresolved                    2.866600e+04
persisted_mvts                2.317493e+06
valid_movement_detections     2.288020e+06
failed_movement_detections    2.947300e+04
raw_coverage_pct              9.752195e+01
persisted_coverage_pct        9.872824e+01
dtype: float64

,airport,raw_mvts,persisted_mvts,valid_detections,raw_coverage_pct,persisted_coverage_pct
0,EDDF,292530,290924,290846,99.424333,99.973189
1,EDDM,212326,210174,210097,98.950199,99.963364
2,EGLL,311270,307442,307404,98.757991,99.987640
3,EHAM,314362,309994,309926,98.588888,99.978064
4,LEBL,234198,231546,231402,98.806138,99.937809
5,LEMD,261538,256520,254894,97.459643,99.366131
6,LFPG,308487,307686,302724,98.131850,98.387317
7,LIRF,181517,179729,174108,95.918289,96.872514
8,LSZH,175085,172327,172206,98.355656,99.929785
9,LTFM,54846,51151,34413,62.744776,67.277277


,month,raw_mvts,persisted_mvts,valid_detections,raw_coverage_pct,persisted_coverage_pct
0,2025-01,127553,126270,125489,98.381849,99.381484
1,2025-02,115921,114573,112216,96.803858,97.942796
2,2025-03,144026,142214,139561,96.899865,98.134502
3,2025-04,142237,140270,137265,96.504426,97.857703
4,2025-05,151000,149352,147090,97.410596,98.485457
5,2025-06,158774,156951,155469,97.918425,99.055756
6,2025-07,163014,161587,160466,98.436944,99.306256
7,2025-08,163247,161806,159914,97.958309,98.830698
8,2025-09,157225,155632,154164,98.053109,99.056749
9,2025-10,161626,159809,158511,98.072711,99.187780


count     2.288020e+06
mean      3.399533e+02
std       4.559948e+02
min      -1.765000e+03
p01      -1.930000e+02
p05      -7.400000e+01
p25      -2.500000e+01
median    2.840000e+02
p75       6.000000e+02
p95       1.047000e+03
p99       1.474000e+03
max       2.159600e+04
dtype: float64

In [7]:
features = pd.read_parquet(FEATURES, columns=["ADEP_mvt", "FLIGHT_ID_mvt", "adsb_movement_status"])
features["NM flight plan"] = features["FLIGHT_ID_mvt"].notna()
pd.crosstab(features["adsb_movement_status"].str.split(":").str[0], features["NM flight plan"], margins=True)

NM flight plan,False,True,All
adsb_movement_status,,,
detected,19998,2268022,2288020
no_sustained_movement,508,28965,29473
unresolved_selection,384,28282,28666
All,20890,2325269,2346159
